In [2]:
import pandas as pd


In [3]:
pip install xlrd

Note: you may need to restart the kernel to use updated packages.


In [4]:
mart_all_data = pd.read_csv('../LTV-Capstone-Project/mart_all_data.csv')
mart_all_data.head(2)

,customer_id,customer_gender,customer_location,customer_tenure_months,transaction_id,transaction_date,produkt_sku,product_category,quantity,avg_price,...,coupon_status,discount_pct,gst_onsale,offline_spend_that_day,online_spend_that_day,transaction_year,transaction_month,transaction_month_str,transaction_day_of_month,transaction_day_of_week
0,15811,F,Illinois,27,25990,2019-04-19,GGOEWCKQ085457,Accessories,1,16.99,...,Clicked,10.0,0.1,4000,1754.92,2019.0,4.0,apr,19.0,friday
1,17999,F,New Jersey,30,25038,2019-04-07,GGOEGBPB081999,Accessories,1,49.99,...,Used,10.0,0.1,2500,2719.46,2019.0,4.0,apr,7.0,sunday


In [5]:
mart_all_data.columns

Index(['customer_id', 'customer_gender', 'customer_location',
       'customer_tenure_months', 'transaction_id', 'transaction_date',
       'produkt_sku', 'product_category', 'quantity', 'avg_price',
       'delivery_charges', 'coupon_status', 'discount_pct', 'gst_onsale',
       'offline_spend_that_day', 'online_spend_that_day', 'transaction_year',
       'transaction_month', 'transaction_month_str',
       'transaction_day_of_month', 'transaction_day_of_week'],
      dtype='object')

In [6]:
# 1. Gross Sales
mart_all_data['gross_sales'] = mart_all_data['quantity'] * mart_all_data['avg_price']

# 2. Discount Amount
mart_all_data['discount_amount'] = mart_all_data['gross_sales'] * (mart_all_data['discount_pct'] / 100)

# 3. Net Subtotal
mart_all_data['net_subtotal'] = mart_all_data['gross_sales'] - mart_all_data['discount_amount']

# 4. Pre-Tax Total (add delivery)
mart_all_data['pre_tax_total'] = mart_all_data['net_subtotal'] + mart_all_data['delivery_charges']

# 5. GST Amount
mart_all_data['gst_amount'] = mart_all_data['pre_tax_total'] * (mart_all_data['gst_onsale'] / 100)

# 6. Final Invoice (what customer paid)
mart_all_data['invoice_value'] = mart_all_data['pre_tax_total'] + mart_all_data['gst_amount']

# 7. Net Revenue (excludes GST remitted to govt)
mart_all_data['net_revenue'] = mart_all_data['pre_tax_total']

In [7]:
historical_ltv = mart_all_data.groupby('customer_id')['net_revenue'].sum().reset_index()
historical_ltv.columns = ['customer_id', 'historical_ltv']
print(historical_ltv.head())

   customer_id  historical_ltv
0        12346         171.693
1        12347       10589.289
2        12348        1339.652
3        12350        1079.929
4        12356        1647.219


In [8]:
mart_all_data.columns

Index(['customer_id', 'customer_gender', 'customer_location',
       'customer_tenure_months', 'transaction_id', 'transaction_date',
       'produkt_sku', 'product_category', 'quantity', 'avg_price',
       'delivery_charges', 'coupon_status', 'discount_pct', 'gst_onsale',
       'offline_spend_that_day', 'online_spend_that_day', 'transaction_year',
       'transaction_month', 'transaction_month_str',
       'transaction_day_of_month', 'transaction_day_of_week', 'gross_sales',
       'discount_amount', 'net_subtotal', 'pre_tax_total', 'gst_amount',
       'invoice_value', 'net_revenue'],
      dtype='object')

In [9]:
pip install lifetimes

Note: you may need to restart the kernel to use updated packages.


In [10]:
from lifetimes.utils import summary_data_from_transaction_data

# Ensure transaction_date is datetime
mart_all_data['transaction_date'] = pd.to_datetime(mart_all_data['transaction_date'])

# Generate summary for LTV modeling
summary = summary_data_from_transaction_data(
    mart_all_data,
    customer_id_col='customer_id',
    datetime_col='transaction_date',
    monetary_value_col='net_revenue',
    observation_period_end=mart_all_data['transaction_date'].max()
)

print(summary.head())


             frequency  recency      T  monetary_value
customer_id                                           
12346              0.0      0.0  107.0           0.000
12347              2.0    223.0  282.0        1163.375
12348              1.0    119.0  192.0         693.731
12350              0.0      0.0   17.0           0.000
12356              0.0      0.0  107.0           0.000


In [11]:
from lifetimes import BetaGeoFitter

bgf = BetaGeoFitter()
bgf.fit(summary['frequency'], summary['recency'], summary['T'])

<lifetimes.BetaGeoFitter: fitted with 1468 subjects, a: 0.58, alpha: 61.18, b: 1.49, r: 0.54>

In [12]:
from lifetimes import GammaGammaFitter

# Only use customers with freq > 0
returning_customers = summary[summary['frequency'] > 0]

ggf = GammaGammaFitter()
ggf.fit(returning_customers['frequency'], returning_customers['monetary_value'])

<lifetimes.GammaGammaFitter: fitted with 734 subjects, p: 0.60, q: 4.81, v: 7983.58>

In [13]:
# Predict expected number of purchases
summary['predicted_purchases_6m'] = bgf.conditional_expected_number_of_purchases_up_to_time(180,
            summary['frequency'], summary['recency'], summary['T'])

# Predict expected average revenue
summary['predicted_avg_value'] = ggf.conditional_expected_average_profit(summary['frequency'], summary['monetary_value'])

# Final LTV
summary['predicted_LTV_6m'] = summary['predicted_purchases_6m'] * summary['predicted_avg_value']

# View top customers by predicted LTV
summary[['predicted_LTV_6m']].sort_values(by='predicted_LTV_6m', ascending=False).head()

,predicted_LTV_6m
customer_id,
15311,21033.926265
14606,17049.487254
14911,15974.121923
17841,14981.066823
13089,9469.652429


In [14]:
summary.head(2)

,frequency,recency,T,monetary_value,predicted_purchases_6m,predicted_avg_value,predicted_LTV_6m
customer_id,,,,,,,
12346,0.0,0.0,107.0,0.000,0.486163,1265.594052,615.284554
12347,2.0,223.0,282.0,1163.375,0.867946,1240.987166,1077.109890


In [15]:
# Merge with demographics or customer segments
mart_customers=pd.read_csv('../LTV-Capstone-Project/mart_customers.csv')
ltv_final = summary.reset_index().merge(mart_customers, on='customer_id', how='left')

# Export to CSV for Tableau
ltv_final.to_csv("Customer_LTV_Summary.csv", index=False)

In [16]:
import os
print(os.getcwd())

c:\Users\dbyst\OneDrive\Desktop\Neue Fische Bootcamp\LTV-Capstone-Project


In [17]:
ltv_final.head(2)

,customer_id,frequency,recency,T,monetary_value,predicted_purchases_6m,predicted_avg_value,predicted_LTV_6m,gender,location,tenure_months
0,12346,0.0,0.0,107.0,0.000,0.486163,1265.594052,615.284554,F,New York,31
1,12347,2.0,223.0,282.0,1163.375,0.867946,1240.987166,1077.109890,M,New York,20


In [18]:
ltv_final.describe()

,customer_id,frequency,recency,T,monetary_value,predicted_purchases_6m,predicted_avg_value,predicted_LTV_6m,tenure_months
count,1468.000000,1468.000000,1468.000000,1468.000000,1468.000000,1468.000000,1468.000000,1468.000000,1468.000000
mean,15314.386240,1.185286,64.358992,208.651226,639.936538,0.690629,1263.838591,877.776033,25.912125
std,1744.000367,2.235245,97.383296,104.677684,1331.197205,0.760671,227.004700,1174.402608,13.959667
min,12346.000000,0.000000,0.000000,0.000000,0.000000,0.000015,564.927071,0.035771,2.000000
25%,13830.500000,0.000000,0.000000,128.000000,0.000000,0.293539,1172.455841,363.268275,14.000000
50%,15300.000000,0.500000,0.500000,222.000000,3.836500,0.448239,1265.594052,553.572812,26.000000
75%,16882.250000,2.000000,121.000000,294.250000,812.452250,0.831766,1265.594052,1055.716632,38.000000
max,18283.000000,33.000000,358.000000,364.000000,19334.968000,10.239648,3738.084777,21033.926265,50.000000


In [19]:
ltv_final.columns

Index(['customer_id', 'frequency', 'recency', 'T', 'monetary_value',
       'predicted_purchases_6m', 'predicted_avg_value', 'predicted_LTV_6m',
       'gender', 'location', 'tenure_months'],
      dtype='object')